# Ordered Logistic Regression Results for Adoption Predictors of Indigenous and Modern Knowledge in Rangeland Management Practices, Northern Kenya: Exploration with `mlcroissant`
This notebook provides a step-by-step guide for loading and exploring the FAIR² dataset using the `mlcroissant` library. We follow the Croissant schema and reference all entities by their `@id`, enabling robust and reproducible data exploration.

### Dataset Source
The dataset source is provided via a Croissant schema URL.

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Define the dataset Croissant schema URL
croissant_url = "https://sen.science/doi/10.71728/senscience.y7m0-f273/fair2.json"

# Load the dataset metadata
dataset = mlc.Dataset(croissant_url)
metadata = dataset.metadata

print(f"Dataset Name: {metadata.name}")
print(f"Description: {metadata.description}\n")
print(f"Identifier: {metadata.identifier}")
print(f"Published: {metadata.datePublished}")
print(f"Spatial Coverage: {metadata.spatialCoverage}")
print(f"License: {metadata.license}")

## 2. Data Overview
Review available record sets, fields, and their IDs.

We list the available record sets and, for each, enumerate their `@id` and contained fields/columns (by their `@id`s).

In [ ]:
# Get a list of available record sets and their fields, referencing by @id
record_sets = dataset.record_sets.keys()

print("Available Record Sets (by @id):")
for rs_id in record_sets:
    record_set = dataset.record_sets[rs_id]
    print(f"  - Record set @id: {rs_id}")
    if hasattr(record_set, 'fields'):
        print("    Fields (by @id):")
        for f in record_set.fields:
            print(f"      - {f['@id'] if '@id' in f else f}")
    if hasattr(record_set, 'columns'):
        print("    Columns (by @id):")
        for c in record_set.columns:
            print(f"      - {c['@id'] if '@id' in c else c}")
    print("")

## 3. Data Extraction
Load data from each record set into a DataFrame for analysis. Use the record set and field `@id`s from the overview above.

For example, let's extract all available record sets. Adjust the `record_sets_to_load` variable if only interested in specific ones.

In [ ]:
# List all record set @id's (from the previous cell output)
record_sets_to_load = list(record_sets)
dataframes = {}

for record_set_id in record_sets_to_load:
    records = list(dataset.records(record_set=record_set_id))
    if len(records) > 0:
        dataframes[record_set_id] = pd.DataFrame(records)
        print(f"Loaded record set {record_set_id} with {len(records)} records. Columns:")
        print(dataframes[record_set_id].columns.tolist())
        print("")
    else:
        print(f"No records found for record set {record_set_id}.")

# For illustration, preview the first few rows of the first non-empty DataFrame
if dataframes:
    example_record_set_id = list(dataframes.keys())[0]
    print(f"Example data from record set {example_record_set_id}:")
    display(dataframes[example_record_set_id].head())
else:
    print("No record sets could be loaded into DataFrames.")

## 4. Exploratory Data Analysis (EDA)
Now, let's process the records for one of the main record sets.

We'll demonstrate common data processing steps: filter on numeric values, normalize a field, and group by a categorical field. Use the correct `@id`s for fields/columns. You may need to adjust the field IDs based on the actual data (see previous code outputs).

In [ ]:
# Example: Assume the first non-empty DataFrame corresponds to an analysis dataset
if dataframes:
    record_set_id = example_record_set_id  # Use previously loaded variable
    df = dataframes[record_set_id]

    print(f"Columns in record set {record_set_id}:")
    print(df.columns.tolist())

    # Attempt to pick a numeric field by scanning DataFrame dtypes
    numeric_field = None
    for col in df.columns:
        if pd.api.types.is_numeric_dtype(df[col]):
            numeric_field = col
            break

    # If no numeric field is found, skip EDA
    if numeric_field:
        print(f"Using numeric field '{numeric_field}' for filtering and normalization.")
        threshold = df[numeric_field].mean()  # Use mean as threshold
        filtered_df = df[df[numeric_field] > threshold]
        print(f"Filtered records with {numeric_field} > {threshold:.2f}:")
        display(filtered_df.head())

        norm_col = f"{numeric_field}_normalized"
        filtered_df[norm_col] = (filtered_df[numeric_field] - filtered_df[numeric_field].mean()) / filtered_df[numeric_field].std()
        print(f"Normalized '{numeric_field}' for filtered records (z-score):")
        display(filtered_df[[numeric_field, norm_col]].head())

        # Attempt to group by the first non-numeric, non-NA column
        group_field = None
        for col in df.columns:
            if not pd.api.types.is_numeric_dtype(df[col]) and df[col].notna().any():
                group_field = col
                break
        if group_field:
            grouped_df = filtered_df.groupby(group_field)[numeric_field].mean().reset_index()
            print(f"Grouped data by '{group_field}', showing mean of '{numeric_field}':")
            display(grouped_df.head())
        else:
            print("No suitable categorical group field found for grouping.")
    else:
        print("No numeric fields found in this record set.")
else:
    print("No DataFrame available for EDA.")

## 5. Visualization
Visualize data distributions or relationships between fields in the dataset.

We generate a histogram for the example numeric field, and a boxplot grouped by the identified group field if available.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

if dataframes and 'numeric_field' in locals() and numeric_field:
    plt.figure(figsize=(8,4))
    sns.histplot(df[numeric_field].dropna(), kde=True, bins=30)
    plt.title(f"Distribution of '{numeric_field}' in record set {record_set_id}")
    plt.xlabel(numeric_field)
    plt.ylabel("Frequency")
    plt.show()

    # Boxplot by group_field
    if 'group_field' in locals() and group_field:
        plt.figure(figsize=(10,5))
        sns.boxplot(x=group_field, y=numeric_field, data=df)
        plt.title(f"Boxplot of '{numeric_field}' by '{group_field}'")
        plt.xlabel(group_field)
        plt.ylabel(numeric_field)
        plt.xticks(rotation=45)
        plt.show()
else:
    print("Not enough data for visualization.")

## 6. Conclusion
In this notebook, we've loaded and explored the FAIR² dataset using the `mlcroissant` library, referencing all entities by their `@id` fields for reproducibility. We listed available record sets and fields, extracted data into pandas DataFrames, and carried out basic exploratory data analysis and visualization. These steps provide a foundation for further analysis or modeling using this dataset.